# 0 Introduction

Import libraries and load the dataset into a dataframe for analysis.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("student-mat.csv", sep=";")


# 1 Exploratory Data Analysis
## Summary Statistics
- We calculated summary statistics to quickly understand the grade distribution before creating visual representations. The results are calculated below.

In [ ]:
finalGrades=df["G3"].values

# Calculate summary stats
mean = np.mean(finalGrades)
median = np.median(finalGrades)
standardDeviation = np.std(finalGrades)

print(f"Summary stastics:\n\tMean: {mean:.3f}\n\tMedian: {median:.3f}\n\tStandard Deviation: {standardDeviation:.3f}")

## Visualizations

 ### Distribution of final gades - Histogram
- We constructed a histogram to visualize the distribution of final grades for all students

In [ ]:
sns.set_style("dark")  # adds seaborn style to charts, eg. grid
plt.style.use("dark_background")  # inverts colors to dark theme
histogram = sns.displot(data=df, x="G3")
plt.xlabel("Final Grade")
plt.ylabel("Number of Students")


# sns.set(style="ticks", context="talk")
# plt.style.use("dark_background")




- The grade distribution appears to be bimodal, with a bell shaped cluster centered around the median and a cohort of students scoring zero points.
---

#### Distribution of failures

In [ ]:
sns.barplot(x = df["failures"].map({0:"0", 1:"1", 2:"2", 3:"3+"}), y = df["G3"], order=["0", "1", "2", "3+"])
plt.xlabel("Number of failed classes")
plt.ylabel("Final Grade")
plt.title("Mean Final Grade by Number of Classes Failed")

- Most students have zero failures, there appears to be a negative correlation between the number of failures and the number of students receving that number of failures. Or the number of students decreases as the number of failures increases.

#### Mean final grade and intent on higher education

In [ ]:
sns.barplot(data=df, x = "higher", y = "G3")
plt.title("Mean Final Grade by Intent on Higher Education")
plt.xlabel("Intend to pursue highed education")
plt.ylabel("Final grade")

- Students who intend to pursue higher education consistently out perform the students who do not. Of those intending to pursue higher education also have a much lower variability in final grade.
- However 375 of the students intend to pursue higher education, leaving 20 who do not. This is not a statistically representative sample and should be discarded for training.


In [ ]:
# Rearrange dataframe ordering by final grade rather than student ID
# Uses alpha transpaency to blend each point based upon either semester one or semester two

melted_df = df.melt(
    id_vars=["G3"],
    value_vars=["G1", "G2"],
    var_name="Grade_Type",
    value_name="Grade_Value",
)

# Single scatterplot call using hue
sns.scatterplot(
    data=melted_df, x="Grade_Value", y="G3", hue="Grade_Type", alpha=0.7
)

plt.xlabel("Semesterly Grade")
plt.ylabel("Final Grade")
plt.title("Semesters One & Two vs Final Grade")
plt.show()

There appears to be a positive correlation between semesterly grades with final grades.

### Comparison of Schools

In [ ]:
gpFinal = df[df["school"] == "GP"]["G3"]
msFinal = df[df["school"] == "MS"]["G3"]
print(f"Gabriel Pereira Statistics:\n\t- median final grade: {np.median(gpFinal)}\n\t- mean final grade: {np.mean(gpFinal)}\n\t- standard deviation of final grades: {np.std(gpFinal)}\n\t- sample size: {gpFinal.count()}")
print(f"Mousinho da Silveira:\n\t- median final grade: {np.median(msFinal)}\n\t- mean final grade: {np.mean(msFinal)}\n\t- standard deviation of final grades: {np.std(msFinal)}\n\t- sample size: {msFinal.count()}")

graph = sns.barplot(x=df["school"].map({"GP": "Gabriel Pereira", "MS": "Mousinho da Silveira"}), y=df["G3"])
plt.ylabel("Final Grade")
plt.xlabel("School")
plt.title("Mean Final Grade by School")
plt.show()

#### Mean final grade for each category by attribute
- This loop iterates through all the attributes then groups the data into subsets to finds the mean for G3 for of the possible values.
- From a quick glance, we can see that most attibute values correlate very little with G3 scores except for Absences and Student Intent to pursue Higher Education.

In [ ]:
for attribute in df.columns:
    if attribute == "G3" or attribute == "G1" or attribute == "G2":
        continue
    print(df.groupby(attribute)["G3"].mean().to_string(float_format="{:.3f}".format) + "\n")

# 2 Data Preprocessing
- We normalized the education of the parents to account for different family structures. For two parent households we average the parents education and in single parent households the value of the custodial parent is assigned.
- True/False categories were encoded strings, "yes" or "no" which are not numeric. We implemented a function to translate these to a binary 1 or 0 which can be correlated with grades.
	- Categories include: extracurricular activities, paid supplemental classes, higher education goals, school support, family support, and nursery attendance.
- The normalization function is displayed below:

In [ ]:
# categories in data set which have true/false values
binaryCategories = ["activities", "paid", "higher", "schoolsup", "famsup", "nursery", "internet", "romantic"]
def normalizeParentalEducation(row):
	if row["Pstatus"] == "T":
		return (row["Fedu"] + row["Medu"])/2
	elif row["guardian"] == "mother":
		return row["Medu"]
	elif row["guardian"] == "father":
		return row["Fedu"]
	else:
		return 0

def tfToBinary(row, cat):
	if cat in binaryCategories:
		if type(row[cat]) == type("string"):
			if row[cat].strip() == "no":
				return 0
			elif row[cat].strip() == "yes":
					return 1
	
#applu normalization
df["normParentEdu"] = df.apply(normalizeParentalEducation, axis=1)
# df["paid"] = df.apply(tfToBinary, categories=["paid"], axis=1)
for cat in binaryCategories:
	df[cat] = df.apply(tfToBinary, cat=cat, axis=1)
# discard input categories after normalization
df = df.drop(columns=["Medu","Fedu"]) 

- After normalizing the data set, we performed a Pearson's correlation analysis of each numerical category against the the final grade. We used this to select the categories to train on in subsequent steps.

In [ ]:
correlations = df.corr(method="pearson", numeric_only=True)["G3"]
print(np.abs(correlations).sort_values(ascending=False).to_string(float_format="{:.3f}".format))

- Our correlation analysis found that the preceeding semester averages had the strongest correlation with the final grade at 90% and 80% respectively. This confirms the trend displayed by the prior graph.
- The number of past failures displayed a negative correlation of 36%. Our normalized parental education had an 18.82% correlation, the remaining attributes had diminishing correlation with final grades.
---

# 3 Model Training and Performance Evalutaion
## Linear Regression
- We used the sklearn machine learning Python library because of the pre-built tools it offers
- We used a linear regression model based upon semester one and two grades.
  - Reasons:
    - G1 and G2 showed the highest correlations to G3.
    - No need to complicate the model and potentially overfit the data. 
- We split our data set 80-20% for the training and testing sets.

In [ ]:
inputTrainSet, inputTestSet, finalGradeTrainSet, finalGradeTestSet = train_test_split(df[["G1", "G2"]], df["G3"], test_size=0.2, random_state=42)
model = LinearRegression().fit(inputTrainSet, finalGradeTrainSet)

finalGradePredictions = model.predict(inputTestSet)

print(f"R^2: {r2_score(finalGradeTestSet, finalGradePredictions): .3f}")
print(f"MSE: {mean_squared_error(finalGradeTestSet, finalGradePredictions): .3f}")

# Pretty print weights of linear regression
equation = f"G3 = {model.intercept_:.3f} + "
for col, coef in zip(inputTestSet.columns, model.coef_):
	equation += f"{coef:.3f} * {col}"

print(f"Regression Equation:")
print(equation)



## Decision Tree
In order to make a more efficient model we categorized the final grades into: 
- Low Grade(0-6)
- Medium Grade(7-13)
- High Grade (14-20) 
### Model Choices:
- criterion = "gini": it is generally faster to calculate than entropy
- splitter = "best": we want the split to be mathematically optimal
- max_depth = 4: from trial and error
- min_samples_split = 2: the defualt value for DecisionTreeClassifier
- min_sample_lead = 10: from trial and error
- random_state = 67: arbitraty, can be any integer
### Results:
- Without G2 and G1 as features,
    - 69.6% accuracy
- With G2 and G1 as features
    - 89.9% accuracy
### Drawbacks:
- The model can only predict a students grade by classifing it into a range of values
- If a more exact prediciton is needed then this tree will not be a very helpful model 



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Categorize final grades into bands for classification
band_edges = [-1, 6, 13, 20]
band_names = ["Low Grade", "Medium Grade", "Good Grade"]
df_cut_g3 = pd.cut(df["G3"], bins=band_edges, labels=band_names)

# Remove G3, G1, and G2 from the input features for classification
df_no_g3 = df.drop(columns=["G3","G1","G2"])
#df_no_g3 = df.drop(columns=["G3"])

# Get the training and testing sets for classification
inputTrainSet_2, inputTestSet_2, finalGradeTrainSet_2, finalGradeTestSet_2 = train_test_split(df_no_g3, df_cut_g3, test_size=0.2, random_state=67)
categories = inputTrainSet_2.select_dtypes(exclude="number").columns.tolist()
numerics = inputTrainSet_2.select_dtypes(include="number").columns.tolist()

# Create a pipeline for preprocessing and classification
# DecidionTreeClassifier requries all input features to be numeric 
# OneHotEncoder converts categorical features into numeric features
column_transformer = ColumnTransformer(
    [("categorize", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                           sparse_output=False), categories)],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
pipeline = Pipeline([("preprocessor", column_transformer),
                     ("classifier",
                      DecisionTreeClassifier(criterion="gini", splitter="best", max_depth=4, min_samples_split=2, min_samples_leaf=10, random_state=67))]
                    )
# Train the model using the training data
pipeline.fit(inputTrainSet_2, finalGradeTrainSet_2)

# Evaluate the model and print feature importances
tree = pipeline.named_steps["classifier"]
features = pipeline.named_steps["preprocessor"].get_feature_names_out()
important_features = pd.Series(pipeline.named_steps["classifier"].feature_importances_, index=features).sort_values(ascending=False)
for feature in important_features.index:
    if important_features[feature] > 0:
        print(f"{feature:>12}: {important_features[feature]: .3f}")
print(f"Decision Tree Classifier Accuracy: {pipeline.score(inputTestSet_2, finalGradeTestSet_2): .3f}")

In [ ]:
##############################################################
# Same process as above but with G1 and G2 as input features #
##############################################################
df_no_g3 = df.drop(columns=["G3"])

# Get the training and testing sets for classification
inputTrainSet_2, inputTestSet_2, finalGradeTrainSet_2, finalGradeTestSet_2 = train_test_split(df_no_g3, df_cut_g3, test_size=0.2, random_state=67)
categories = inputTrainSet_2.select_dtypes(exclude="number").columns.tolist()
numerics = inputTrainSet_2.select_dtypes(include="number").columns.tolist()

# Create a pipeline for preprocessing and classification
# DecidionTreeClassifier requries all input features to be numeric 
# OneHotEncoder converts categorical features into numeric features
column_transformer = ColumnTransformer(
    [("categorize", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                           sparse_output=False), categories)],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
pipeline = Pipeline([("preprocessor", column_transformer),
                     ("classifier",
                      DecisionTreeClassifier(criterion="gini", splitter="best", max_depth=4, min_samples_split=2, min_samples_leaf=10, random_state=67))]
                    )
# Train the model using the training data
pipeline.fit(inputTrainSet_2, finalGradeTrainSet_2)

# Evaluate the model and print feature importances
tree2 = pipeline.named_steps["classifier"]
features = pipeline.named_steps["preprocessor"].get_feature_names_out()
important_features = pd.Series(tree2.feature_importances_, index=features).sort_values(ascending=False)
for feature in important_features.index:
    if important_features[feature] > 0:
        print(f"{feature:>12}: {important_features[feature]: .3f}")
print(f"Decision Tree Classifier Accuracy: {pipeline.score(inputTestSet_2, finalGradeTestSet_2): .3f}")

## 4. Model Performance Visualization
- Ipsum Lorem

In [ ]:
resultDf = pd.DataFrame({"finalGradePredictions" : finalGradePredictions, 'finalGradeTestSet' : finalGradeTestSet, "absDifference" : np.abs(finalGradePredictions - finalGradeTestSet)})

sns.scatterplot(resultDf, x="finalGradePredictions", y="finalGradeTestSet")
plt.errorbar(
    x=resultDf["finalGradePredictions"],
    y=resultDf["finalGradeTestSet"],
    yerr=resultDf["absDifference"],  # Use yerr=resultDf["absDifference"] for vertical error bars
    fmt="none",  # Don't plot new markers on top of seaborn scatter
    ecolor="lightgray",  # Light color so it doesn't obscure points
    alpha=0.6,
    capsize=3,  # Adds small horizontal caps to the end of error bars
)
plt.xlabel("Predicted Final Grade")
plt.ylabel("Actual Final Grade")
plt.title("Predictions vs Actuals with Absolute Error")
plt.show()

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(12, 8))
plot_tree(tree, feature_names=features, class_names=band_names, filled=True, rounded=True, fontsize=11)
plt.title("Decision Tree Classifier for Final Grade Bands (Excluding G1 and G2)")
plt.show()

plt.figure(figsize=(12, 8))
plot_tree(tree2, feature_names=features, class_names=band_names, filled=True, rounded=True, fontsize=11)
plt.title("Decision Tree Classifier for Final Grade Bands (Including G1 and G2)")
plt.show()


# References
- https://www.geeksforgeeks.org/machine-learning/best-python-libraries-for-machine-learning/
- https://realpython.com/python-seaborn/
- https://seaborn.pydata.org/tutorial/distributions.html
- https://www.geeksforgeeks.org/python/use-error-bars-in-a-matplotlib-scatter-plot/
- https://scikit-learn.org/stable/modules/linear_model.html
- https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
- https://medium.com/@sahin.samia/scikit-learn-pipelines-explained-streamline-and-optimize-your-machine-learning-processes-f17b1beb86a4
- https://www.youtube.com/watch?v=R15Rm_aW1jM
- https://www.youtube.com/watch?v=xIqX1dqcNbY
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html